In [0]:
# DBTITLE 1,Importação de Bibliotecas e Funções
from pyspark.sql.functions import (
    col, sum as _sum, count, round, current_timestamp
)

# Definição dos nomes do Catálogo e Schema
CATALOG = "workspace"
SCHEMA = "default"

# DBTITLE 2,1. Dimensão Clientes (dim_customers)
df_silver_customers = spark.table(f"{CATALOG}.{SCHEMA}.silver_customers")

df_dim_customers = (
    df_silver_customers
    .select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_city"),
        col("customer_state")
    )
    .distinct()
    .withColumn("_created_at", current_timestamp())
)

df_dim_customers.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_customers")
print("✓ Tabela dim_customers criada com sucesso!")

# DBTITLE 3,2. Dimensão Produtos (dim_products)
df_silver_products = spark.table(f"{CATALOG}.{SCHEMA}.silver_products")

df_dim_products = (
    df_silver_products
    .select(
        col("product_id"),
        col("product_category_name")
    )
    .distinct()
    .withColumn("_created_at", current_timestamp())
)

df_dim_products.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_products")
print("✓ Tabela dim_products criada com sucesso!")

# DBTITLE 4,3. Tabela Fato Pedidos (fact_orders)
df_orders = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
df_items = spark.table(f"{CATALOG}.{SCHEMA}.silver_order_items")
df_payments = spark.table(f"{CATALOG}.{SCHEMA}.silver_payments")

# Agregação de itens por pedido (utilizando order_item_id)
df_items_agg = (
    df_items
    .groupBy("order_id")
    .agg(
        _sum("price").alias("total_products_value"),
        _sum("freight_value").alias("total_freight_value"),
        count("order_item_id").alias("total_items")
    )
)

# Agregação de pagamentos por pedido
df_payments_agg = (
    df_payments
    .groupBy("order_id")
    .agg(
        _sum("payment_value").alias("total_paid_value")
    )
)

# Consolidação da Tabela Fato Pedidos
df_fact_orders = (
    df_orders
    .join(df_items_agg, "order_id", "left")
    .join(df_payments_agg, "order_id", "left")
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_purchase_timestamp"),
        col("order_delivered_customer_date"),
        col("delivery_time_days"),
        col("is_delayed"),
        col("total_items"),
        round(col("total_products_value"), 2).alias("total_products_value"),
        round(col("total_freight_value"), 2).alias("total_freight_value"),
        round(col("total_paid_value"), 2).alias("total_paid_value")
    )
    .withColumn("_created_at", current_timestamp())
)

df_fact_orders.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.fact_orders")
print("✓ Tabela fact_orders criada com sucesso!")

print("\n--- Pipeline Medallion Concluído com Sucesso! ---")

✓ Tabela dim_customers criada com sucesso!
✓ Tabela dim_products criada com sucesso!
✓ Tabela fact_orders criada com sucesso!

--- Pipeline Medallion Concluído com Sucesso! ---
